# CI/CD Pipeline: Diabetes Readmission Prediction
**AAI-540 Group 5**

This notebook builds an automated SageMaker Pipeline (DAG) that preprocesses data, trains an XGBoost model, evaluates it, and registers it to the Model Registry only if it meets a quality threshold. Every pipeline run is versioned and repeatable, which is the basis of our CI/CD process.

**Pipeline flow:** Preprocess → Train → Evaluate → Condition Check → Register Model

In [59]:
%pip install -q "sagemaker>=2.200,<3"

Note: you may need to restart the kernel to use updated packages.


## 1. Environment Setup
We pin the SageMaker SDK below v3 for compatibility, then create a `PipelineSession`. Steps built with a pipeline session are defined but not run right away. They run only when the pipeline itself executes.

In [60]:
# Standard libraries
import os
import json
import time
from time import sleep

# Data
import pandas as pd

# AWS / SageMaker core
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics

# Processing
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

# Pipeline building blocks
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString, ParameterInteger, ParameterFloat
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.fail_step import FailStep

# Session setup
region = boto3.Session().region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
bucket = pipeline_session.default_bucket()
prefix = "diabetes-readmission/pipeline"
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")

print(region, role, bucket, sep="\n")
print("SageMaker SDK:", sagemaker.__version__)

os.chdir(os.path.expanduser("~/diabetes-readmission-prediction/notebooks/setup"))
print("Working dir:", os.getcwd())

us-east-1
arn:aws:iam::135844594927:role/LabRole
sagemaker-us-east-1-135844594927
SageMaker SDK: 2.257.6
Working dir: /home/sagemaker-user/diabetes-readmission-prediction/notebooks/setup


## 2. Locate Input Data
We find the cleaned diabetes dataset in S3 that our data ingestion step produced. This becomes the input to the pipeline's Preprocess step.

In [61]:
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if "diabet" in key.lower() and key.endswith((".csv", ".parquet")):
            print(f"s3://{bucket}/{key}  ({obj['Size']/1e6:.1f} MB)")

s3://sagemaker-us-east-1-135844594927/DiabetesReadmissionPipeline/luvlcm8i30e5/PreprocessDiabetesData/output/test/test.csv  (7.8 MB)
s3://sagemaker-us-east-1-135844594927/DiabetesReadmissionPipeline/luvlcm8i30e5/PreprocessDiabetesData/output/train/train.csv  (37.1 MB)
s3://sagemaker-us-east-1-135844594927/DiabetesReadmissionPipeline/luvlcm8i30e5/PreprocessDiabetesData/output/validation/validation.csv  (7.8 MB)
s3://sagemaker-us-east-1-135844594927/carepath-analytics/processed/diabetes_processed.csv  (29.5 MB)
s3://sagemaker-us-east-1-135844594927/carepath-analytics/raw/diabetic_data.csv  (19.2 MB)


## 3. Inspect Processed Data
We check the shape, columns, and target distribution of the processed dataset so the pipeline's preprocessing script handles it correctly.

In [62]:
input_data_uri = f"s3://{bucket}/carepath-analytics/processed/diabetes_processed.csv"

df = pd.read_csv(input_data_uri)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nTarget candidates:")
for col in df.columns:
    if "readmit" in col.lower() or "target" in col.lower() or "label" in col.lower():
        print(col, df[col].value_counts().to_dict())

Shape: (101766, 57)

Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'readmitted_30d', 'admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'diag_3_group']

Dtypes:
 object    43


## 4. Preprocessing Script
This script runs inside the pipeline's Processing step. It:
- Drops ID columns, the leakage column (`readmitted`), mostly-empty columns (`weight`, `payer_code`), raw diagnosis codes (we keep the grouped versions), and numeric ID codes (we keep the readable names)
- Fills missing categorical values with "Unknown" and one-hot encodes them
- Splits by **patient** (70/15/15) so the same patient never appears in both train and test, which prevents data leakage
- Saves train, validation, and test CSVs with the label in the first column and no header, the format SageMaker's built-in XGBoost expects

In [63]:
%%writefile preprocess.py
import os
import numpy as np
import pandas as pd

BASE = "/opt/ml/processing"
TARGET = "readmitted_30d"
DROP_COLS = [
    "encounter_id", "readmitted", "weight", "payer_code",
    "diag_1", "diag_2", "diag_3",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "examide", "citoglipton",
]

if __name__ == "__main__":
    df = pd.read_csv(f"{BASE}/input/diabetes_processed.csv")
    print("Input shape:", df.shape)

    # Patient-level split to prevent leakage
    rng = np.random.default_rng(42)
    patients = df["patient_nbr"].unique()
    rng.shuffle(patients)
    n = len(patients)
    train_p = set(patients[: int(0.70 * n)])
    val_p = set(patients[int(0.70 * n): int(0.85 * n)])
    split = df["patient_nbr"].map(
        lambda p: "train" if p in train_p else ("validation" if p in val_p else "test")
    )

    # Clean and encode
    df = df.drop(columns=DROP_COLS + ["patient_nbr"], errors="ignore")
    cat_cols = df.select_dtypes(include="object").columns
    df[cat_cols] = df[cat_cols].replace("?", "Unknown").fillna("Unknown")
    df = pd.get_dummies(df, columns=cat_cols, dtype=int)

    # Label first for built-in XGBoost
    df = df[[TARGET] + [c for c in df.columns if c != TARGET]]

    for name in ["train", "validation", "test"]:
        part = df[split == name]
        out_dir = f"{BASE}/{name}"
        os.makedirs(out_dir, exist_ok=True)
        part.to_csv(f"{out_dir}/{name}.csv", header=False, index=False)
        print(f"{name}: {part.shape}, positive rate {part[TARGET].mean():.3f}")

Overwriting preprocess.py


## 5. Pipeline Parameters and Processing Step
**Parameters** make the pipeline reusable. A CI/CD run can pass a new data path, instance type, or quality threshold without any code changes.

**Processing step:** runs `preprocess.py` in a managed scikit-learn container. It reads the processed dataset from S3 and writes train, validation, and test splits back to S3 for the next steps. Caching skips this step on reruns when its inputs haven't changed, which saves Learner Lab budget.

In [64]:
# Pipeline parameters
input_data = ParameterString(name="InputData", default_value=input_data_uri)
processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.m5.large")
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.large")
auc_threshold = ParameterFloat(name="AUCThreshold", default_value=0.65)
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")

cache_config = CacheConfig(enable_caching=True, expire_after="PT12H")

# Processor
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-preprocess",
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    code="preprocess.py",
    inputs=[ProcessingInput(source=input_data, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
)

step_process = ProcessingStep(
    name="PreprocessDiabetesData",
    step_args=processor_args,
    cache_config=cache_config,
)

print("Processing step defined:", step_process.name)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Processing step defined: PreprocessDiabetesData


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 6. Training Step
Trains SageMaker's built-in XGBoost on the train split and scores each round on the validation split.
- `scale_pos_weight ≈ 8` makes up for the class imbalance (about 8 negatives for every positive)
- `eval_metric = auc` with early stopping halts training when validation AUC stops improving
- The step reads the S3 outputs of the Processing step directly, so SageMaker knows to run Processing first. This creates the first edge in the DAG.

In [65]:
model_output_path = f"s3://{bucket}/{prefix}/model"

xgb_image = image_uris.retrieve(framework="xgboost", region=region, version="1.7-1")

xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=model_output_path,
    base_job_name="diabetes-xgb-train",
    sagemaker_session=pipeline_session,
)

xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=500,
    max_depth=6,
    eta=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=8,
)

train_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

step_train = TrainingStep(
    name="TrainXGBoostModel",
    step_args=train_args,
    cache_config=cache_config,
)

print("Training step defined:", step_train.name)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Training step defined: TrainXGBoostModel


## 7. Evaluation Script
This script runs inside the pipeline's Evaluation step. It loads the trained model artifact, scores the held-out **test** split, and writes `evaluation.json` with AUC, accuracy, precision, recall, F1, and the confusion matrix.
The JSON uses SageMaker's `binary_classification_metrics` format. That lets the Condition step read the AUC and lets the Model Registry show the metrics next to each model version.

In [66]:
%%writefile evaluate.py
import json
import os
import tarfile

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

BASE = "/opt/ml/processing"

if __name__ == "__main__":
    with tarfile.open(f"{BASE}/model/model.tar.gz") as tar:
        tar.extractall(path=".")
    booster = xgb.Booster()
    booster.load_model("xgboost-model")

    df = pd.read_csv(f"{BASE}/test/test.csv", header=None)
    y_true = df.iloc[:, 0].to_numpy()
    X = xgb.DMatrix(df.iloc[:, 1:].to_numpy())

    y_prob = booster.predict(X)
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics = {
        "binary_classification_metrics": {
            "auc": {"value": float(roc_auc_score(y_true, y_prob)), "standard_deviation": "NaN"},
            "accuracy": {"value": float(accuracy_score(y_true, y_pred)), "standard_deviation": "NaN"},
            "precision": {"value": float(precision_score(y_true, y_pred, zero_division=0)), "standard_deviation": "NaN"},
            "recall": {"value": float(recall_score(y_true, y_pred)), "standard_deviation": "NaN"},
            "f1": {"value": float(f1_score(y_true, y_pred)), "standard_deviation": "NaN"},
            "confusion_matrix": {
                "0": {"0": int(tn), "1": int(fp)},
                "1": {"0": int(fn), "1": int(tp)},
            },
        }
    }

    os.makedirs(f"{BASE}/evaluation", exist_ok=True)
    with open(f"{BASE}/evaluation/evaluation.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))

Overwriting evaluate.py


## 8. Evaluation Step
Runs `evaluate.py` inside the same XGBoost container used for training, so the library versions match. It takes two inputs:
- The **model artifact** from the Training step
- The **test split** from the Processing step

Because it depends on both, SageMaker runs it only after Processing and Training finish. The output `evaluation.json` is registered as a **PropertyFile**, which lets the next step read the AUC value from it.

In [67]:
eval_processor = ScriptProcessor(
    image_uri=xgb_image,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-xgb-eval",
    sagemaker_session=pipeline_session,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

eval_args = eval_processor.run(
    code="evaluate.py",
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
)

step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=eval_args,
    property_files=[evaluation_report],
    cache_config=cache_config,
)

print("Evaluation step defined:", step_eval.name)

Evaluation step defined: EvaluateModel


## 9. Register Model Step
Registers the trained model as a new version in the SageMaker **Model Registry** under the `diabetes-readmission-models` group. Each version stores:
- The model artifact and XGBoost container image
- The evaluation metrics from `evaluation.json`, viewable in the Registry UI
- An approval status (`PendingManualApproval` by default), so a human reviews the model before it's deployed

This step runs only if the model passes the AUC check in the next step.

In [68]:
model_package_group_name = "CarePath-Diabetes-Readmission"

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

model = Model(
    image_uri=xgb_image,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
    description="XGBoost 30-day readmission model trained by CI/CD pipeline",
)

step_register = ModelStep(name="RegisterDiabetesModel", step_args=register_args)

print("Register step defined")

Register step defined


## 10. Condition Step (Quality Gate)
Reads the test AUC from `evaluation.json` and compares it with the `AUCThreshold` parameter.
- **AUC ≥ threshold:** the model is registered in the Model Registry
- **AUC < threshold:** the pipeline stops with a **Fail** step and a clear error message, and nothing is registered

This gate is the core of our CI/CD process. Only models that meet the quality bar move toward deployment.

In [69]:
step_fail = FailStep(
    name="AUCBelowThreshold",
    error_message=Join(
        on=" ",
        values=["Model AUC is below the threshold of", auc_threshold, ". Model not registered."],
    ),
)

cond_auc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="binary_classification_metrics.auc.value",
    ),
    right=auc_threshold,
)

step_cond = ConditionStep(
    name="CheckAUCThreshold",
    conditions=[cond_auc],
    if_steps=[step_register],
    else_steps=[step_fail],
)

print("Condition step defined:", step_cond.name)

Condition step defined: CheckAUCThreshold


## 11. Assemble and Register the Pipeline
We combine all the steps into one SageMaker Pipeline. We don't have to set the order by hand: SageMaker builds the DAG from the data dependencies between steps.

**DAG:** PreprocessDiabetesData → TrainXGBoostModel → EvaluateModel → CheckAUCThreshold → (RegisterDiabetesModel | AUCBelowThreshold)

`upsert()` creates the pipeline the first time and updates it on later runs. This is what lets CI/CD push new versions of the pipeline definition.

In [70]:
pipeline_name = "DiabetesReadmissionPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        input_data,
        processing_instance_type,
        training_instance_type,
        auc_threshold,
        model_approval_status,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

# Check the DAG structure
definition = json.loads(pipeline.definition())
for s in definition["Steps"]:
    print(f"{s['Name']:<25} {s['Type']}")

# Create or update the pipeline in SageMaker
response = pipeline.upsert(role_arn=role)
print("\nPipeline ARN:", response["PipelineArn"])

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


PreprocessDiabetesData    Processing
TrainXGBoostModel         Training
EvaluateModel             Processing
CheckAUCThreshold         Condition



Pipeline ARN: arn:aws:sagemaker:us-east-1:135844594927:pipeline/DiabetesReadmissionPipeline


## 12. Execute the Pipeline
We start a pipeline run using the default parameters. SageMaker runs each step in DAG order and starts managed jobs for Processing, Training, and Evaluation. A full run takes about 15 to 20 minutes.

In [71]:
execution = pipeline.start()
print("Execution ARN:", execution.arn)

execution.wait(delay=30, max_attempts=80)

for step in execution.list_steps():
    print(f"{step['StepName']:<25} {step['StepStatus']}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:us-east-1:135844594927:pipeline/DiabetesReadmissionPipeline/execution/piauumirqsx8
RegisterDiabetesModel-RegisterModel Succeeded
CheckAUCThreshold         Succeeded
EvaluateModel             Succeeded
TrainXGBoostModel         Succeeded
PreprocessDiabetesData    Succeeded


In [72]:
objs = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket):
    for o in page.get("Contents", []):
        if o["Key"].endswith("evaluation.json"):
            objs.append(o)

latest = max(objs, key=lambda o: o["LastModified"])
print("File:", latest["Key"], "|", latest["LastModified"])

body = s3.get_object(Bucket=bucket, Key=latest["Key"])["Body"].read()
print(json.dumps(json.loads(body), indent=2))

File: DiabetesReadmissionPipeline/piauumirqsx8/EvaluateModel/output/evaluation/evaluation.json | 2026-10-14 01:44:10+00:00
{
  "binary_classification_metrics": {
    "auc": {
      "value": 0.6628140408076003,
      "standard_deviation": "NaN"
    },
    "accuracy": {
      "value": 0.6833344386232508,
      "standard_deviation": "NaN"
    },
    "precision": {
      "value": 0.1786898839137645,
      "standard_deviation": "NaN"
    },
    "recall": {
      "value": 0.5146268656716418,
      "standard_deviation": "NaN"
    },
    "f1": {
      "value": 0.2652715802431143,
      "standard_deviation": "NaN"
    },
    "confusion_matrix": {
      "0": {
        "0": 9442,
        "1": 3962
      },
      "1": {
        "0": 813,
        "1": 862
      }
    }
  }
}


In [73]:
latest_exec = sm.list_pipeline_executions(
    PipelineName=pipeline_name, SortBy="CreationTime", SortOrder="Descending"
)["PipelineExecutionSummaries"][0]
exec_arn = latest_exec["PipelineExecutionArn"]
print("Execution:", exec_arn.split("/")[-1], "|", latest_exec["PipelineExecutionStatus"])

steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)["PipelineExecutionSteps"]
for s in steps:
    print(f"{s['StepName']:<35} {s['StepStatus']}")

eval_meta = [s for s in steps if s["StepName"] == "EvaluateModel"][0]["Metadata"]["ProcessingJob"]["Arn"]
job = sm.describe_processing_job(ProcessingJobName=eval_meta.split("/")[-1])
eval_uri = job["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"] + "/evaluation.json"
print("\nEval file:", eval_uri)

b, k = eval_uri.replace("s3://", "").split("/", 1)
print(json.dumps(json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read()), indent=2))

Execution: piauumirqsx8 | Succeeded
RegisterDiabetesModel-RegisterModel Succeeded
CheckAUCThreshold                   Succeeded
EvaluateModel                       Succeeded
TrainXGBoostModel                   Succeeded
PreprocessDiabetesData              Succeeded

Eval file: s3://sagemaker-us-east-1-135844594927/DiabetesReadmissionPipeline/piauumirqsx8/EvaluateModel/output/evaluation/evaluation.json
{
  "binary_classification_metrics": {
    "auc": {
      "value": 0.6628140408076003,
      "standard_deviation": "NaN"
    },
    "accuracy": {
      "value": 0.6833344386232508,
      "standard_deviation": "NaN"
    },
    "precision": {
      "value": 0.1786898839137645,
      "standard_deviation": "NaN"
    },
    "recall": {
      "value": 0.5146268656716418,
      "standard_deviation": "NaN"
    },
    "f1": {
      "value": 0.2652715802431143,
      "standard_deviation": "NaN"
    },
    "confusion_matrix": {
      "0": {
        "0": 9442,
        "1": 3962
      },
      "1": {


In [74]:
logs = boto3.client("logs")
group = "/aws/sagemaker/ProcessingJobs"

streams = logs.describe_log_streams(
    logGroupName=group, logStreamNamePrefix=job_name
)["logStreams"]

for st in streams:
    events = logs.get_log_events(
        logGroupName=group, logStreamName=st["logStreamName"], startFromHead=False, limit=40
    )["events"]
    for e in events:
        print(e["message"])

Traceback (most recent call last):
  File "/opt/ml/processing/input/code/preprocess.py", line 12, in <module>
    df = pd.read_csv(f"{BASE}/input/diabetes_processed.csv")
NameError: name 'pd' is not defined


## 13. Verify Pipeline Results in the Model Registry
We confirm that the pipeline registered a new model version and check its test metrics. Each successful run adds a version to the `diabetes-readmission-models` group, with status `PendingManualApproval`, giving us a versioned, auditable history of every model the pipeline produces.

In [75]:
packages = sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
)["ModelPackageSummaryList"]

for p in packages:
    print(f"Version {p['ModelPackageVersion']:<3} {p['ModelApprovalStatus']:<25} {p['CreationTime']:%Y-%m-%d %H:%M}")

latest_pkg = sm.describe_model_package(ModelPackageName=packages[0]["ModelPackageArn"])
metrics_uri = latest_pkg["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
b, k = metrics_uri.replace("s3://", "").split("/", 1)
latest_metrics = json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())

print("\nLatest model test metrics:")
for name, val in latest_metrics["binary_classification_metrics"].items():
    if name != "confusion_matrix":
        print(f"  {name:<10} {val['value']:.4f}")
print("  confusion  ", latest_metrics["binary_classification_metrics"]["confusion_matrix"])

Version 2   PendingManualApproval     2026-10-14 01:46
Version 1   Approved                  2026-10-03 01:45

Latest model test metrics:
  auc        0.6628
  accuracy   0.6833
  precision  0.1787
  recall     0.5146
  f1         0.2653
  confusion   {'0': {'0': 9442, '1': 3962}, '1': {'0': 813, '1': 862}}


## 14. Results
The pipeline registers each passing model as a new version in the team's Model Registry group, `CarePath-Diabetes-Readmission`. The AUC quality gate is set to 0.65.

| Metric | Quang's model (v1, manual) | Pipeline (v2, automated) |
|---|---|---|
| ROC AUC | 0.684 | 0.663 |
| F1 | 0.264 | 0.265 |
| Recall | 0.725 | 0.515 |
| Precision | 0.161 | 0.179 |

**Notes**
- Hyperparameters match the team's final model (500 rounds, depth 6, learning rate 0.03, subsample 0.9, colsample 0.9).
- The remaining AUC gap comes from preprocessing differences. The pipeline uses all features and a patient-level split. v1 uses 19 selected features.
- The recall gap comes from the decision threshold (0.5 here vs. 0.435 tuned in v1).
- v2 stays at PendingManualApproval, so a human reviews it before deployment.
- An earlier run failed at Preprocess because the script was missing its imports. It was found through CloudWatch logs and fixed, and no downstream steps ran on bad data.